# \# Handwritten Digit Classification using CNN & RNN

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import torchvision
from torchvision.datasets import MNIST 
import torchvision.transforms as transforms

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [3]:
train_set = MNIST(root='./Data', train=True, download=True, transform = transform)
test_set = MNIST(root='./Data', train=False, download=True, transform = transform)

In [4]:
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=32, shuffle = True)

# building CNN

In [5]:

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(

            # 1st Conv block
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # 2nd Conv block
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # 3rd Conv block
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            
            nn.Linear(128 * 3 * 3, 128),
            nn.ReLU(),
            
            nn.Linear(128, 10)  # 10 MNIST classes
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

In [6]:
# cnn model

model = CNN()

#loss, optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
epochs = 5

for epoch in range(epochs):

    model.train()
    train_loss = 0.0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch + 1}/{epochs}, "f"Loss: {train_loss / len(train_loader):.4f}")

Epoch 1/5, Loss: 0.1337


In [ ]:
# evaluation of cnn

total_label = 0
correct_label = 0

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total_label += labels.size(0)
        correct_label += (predicted == labels).sum().item()

print("Total Labels :", total_label)
print("Correct Labels :", correct_label)
print("Accuracy :", (correct_label / total_label) * 100)

# RNN Model

In [ ]:
class RNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=28,
            hidden_size=128,
            num_layers=2,
            batch_first=True
        )

        self.fc = nn.Linear(128, 10)

    def forward(self, x):
        out, hidden = self.rnn(x)
        out = self.fc(out[:, -1, :])
        return out

In [ ]:
rnn_model = RNN()

rnn_optimizer = torch.optim.Adam(rnn_model.parameters())

In [ ]:
epochs = 5

for epoch in range(epochs):

    rnn_model.train()
    train_loss = 0.0

    for images, labels in train_loader:

        rnn_optimizer.zero_grad()

        images = images.squeeze(1)

        outputs = rnn_model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        rnn_optimizer.step()

        train_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/{epochs}, "
        f"Loss: {train_loss / len(train_loader):.4f}"
    )

In [ ]:
# Evaluation of RNN

total_labels = 0
correct_labels = 0

rnn_model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.squeeze(1)
        outputs = rnn_model(images)

        _, predicted = torch.max(outputs, 1)

        total_labels += labels.size(0)
        correct_labels += (predicted == labels).sum().item()

print("Total Labels :", total_labels)
print("Correct Labels :", correct_labels)
print("Accuracy :", (correct_labels / total_labels) * 100)